# Import dataset

In [1]:
import sys
print(sys.executable)

/Users/nicole/data-training/.venv/bin/python


In [1]:
import plotly.express as px
import numpy as np
import json

In [3]:
# read bigquery data into pandas dataframe
import pandas as pd

df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

/var/folders/03/kg6152lx62g3z5rhmtm47_c80000gn/T/ipykernel_43610/1097260367.py:4: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df = pd.read_gbq(


# Identification of peak revenue hours.

## Find the revenue trand in each hour according to the order create time

In [4]:
# Extract out the HOUR from the time frame
df['time'] = df['date_created'].dt.strftime('%H')
hour_sales = df[(df["status"] == 2)].groupby('time')['total'].sum() / 100

fig = px.line(x=hour_sales.index.astype(str), y=hour_sales.values, labels={'x':'Hour', 'y':'Total Sales($AUD)'}, title='Hourly Sales')
fig.show()

Obviously, the peak revenue hour is at 8:00A.M, with a total of 86.97K revenue.

## Distinct the data by year

In [5]:
# Add a year column
df['year'] = df['date_created'].dt.year

hour_sales_per_year = df[df["status"] == 2].groupby(['year', 'time'])['total'].sum() / 100

fig = px.line(hour_sales_per_year.reset_index(), x='time', y='total', color='year',
              labels={'time':'Hour', 'total':'Total Sales($AUD)'}, title='Hourly Sales',
              line_group='year')
fig.show()

## Distinct the data by day of week

In [8]:
df['day_of_week'] = df['date_created'].dt.day_name()

hour_sales_per_weekday = df[df["status"] == 2].groupby(['day_of_week', 'time'])['total'].sum() / 100

fig = px.line(hour_sales_per_weekday.reset_index(), x='time', y='total', color='day_of_week',
              labels={'time':'Hour', 'total':'Total Sales($AUD)'}, title='Hourly Sales',
              line_group='day_of_week')
fig.show()

We can see that the trend are different between weekdays and weekends.
Usually weekdays' peak hour is around 7-8 A.M, while weekends is around 8-9 A.M.

## Extract the order time from the dataset

In [ ]:
from datetime import datetime

def fill_order_time(row):

    items_dict = json.loads(row['items'])

    try:
        order_time_verbose = items_dict['order_time_verbose'].split()[-1] #only get the time
    except KeyError:
        order_time_verbose = 'ASAP'


    if order_time_verbose != 'ASAP':

        time_formats = ["%I:%M", "%I:%M%p", "%I:%M %p"] # different types of time format

        for fmt in time_formats:
            try:
                parsed_time = datetime.strptime(order_time_verbose, fmt)
                out_time = datetime.strftime(parsed_time, '%H:%M')
                return out_time
            except ValueError:
                pass

    else:

        date_created = row['date_created']
        out_time = date_created.strftime('%H:%M')

        return out_time